In [229]:
print("MODEL TRAINING")

MODEL TRAINING


In [230]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
from sklearn.cluster import SpectralClustering
from sklearn.cluster import Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler


In [278]:
df=pd.read_csv('../data/modified_amazon_sales.csv')
df.head()

,cat__Ship Mode_Same Day,cat__Ship Mode_Second Class,cat__Ship Mode_Standard Class,cat__Segment_Corporate,cat__Segment_Home Office,cat__Region_East,cat__Region_South,cat__Region_West,cat__Category_Office Supplies,cat__Category_Technology,...,remainder__Quantity,remainder__Discount,remainder__Profit,remainder__order_year,remainder__order_month,remainder__shipping_days,remainder__City_freq,remainder__State_freq,remainder__Sales_per_Item,remainder__Discount_Impact
0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,2,0.00,41.9136,2016,11,3,51,139,130.9800,0.000000
1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,3,0.00,219.5820,2016,11,3,51,139,243.9800,0.000000
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,...,2,0.00,6.8714,2016,6,4,747,2001,7.3100,0.000000
3,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,5,0.45,-383.0310,2015,10,7,15,383,191.5155,430.909875
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,2,0.20,2.5164,2015,10,7,15,383,11.1840,4.473600


In [232]:
cols_to_drop = [
    "cat__Ship Mode_Same Day",
    "cat__Ship Mode_Second Class",
    "cat__Ship Mode_Standard Class",
    "remainder__order_year",
    "remainder__City_freq",
    "remainder__State_freq",
    "remainder__order_month",
    'remainder__shipping_days'
]

df = df.drop(columns=cols_to_drop)

In [233]:
df.head()


,cat__Segment_Corporate,cat__Segment_Home Office,cat__Region_East,cat__Region_South,cat__Region_West,cat__Category_Office Supplies,cat__Category_Technology,cat__Sub-Category_Appliances,cat__Sub-Category_Art,cat__Sub-Category_Binders,...,cat__Sub-Category_Phones,cat__Sub-Category_Storage,cat__Sub-Category_Supplies,cat__Sub-Category_Tables,remainder__Sales,remainder__Quantity,remainder__Discount,remainder__Profit,remainder__Sales_per_Item,remainder__Discount_Impact
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,261.9600,2,0.00,41.9136,130.9800,0.000000
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,731.9400,3,0.00,219.5820,243.9800,0.000000
2,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,14.6200,2,0.00,6.8714,7.3100,0.000000
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,957.5775,5,0.45,-383.0310,191.5155,430.909875
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,22.3680,2,0.20,2.5164,11.1840,4.473600


In [234]:
numerical_cols=[]
categorical_cols=[]

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()  
print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

Numerical Columns: ['cat__Segment_Corporate', 'cat__Segment_Home Office', 'cat__Region_East', 'cat__Region_South', 'cat__Region_West', 'cat__Category_Office Supplies', 'cat__Category_Technology', 'cat__Sub-Category_Appliances', 'cat__Sub-Category_Art', 'cat__Sub-Category_Binders', 'cat__Sub-Category_Bookcases', 'cat__Sub-Category_Chairs', 'cat__Sub-Category_Copiers', 'cat__Sub-Category_Envelopes', 'cat__Sub-Category_Fasteners', 'cat__Sub-Category_Furnishings', 'cat__Sub-Category_Labels', 'cat__Sub-Category_Machines', 'cat__Sub-Category_Paper', 'cat__Sub-Category_Phones', 'cat__Sub-Category_Storage', 'cat__Sub-Category_Supplies', 'cat__Sub-Category_Tables', 'remainder__Sales', 'remainder__Quantity', 'remainder__Discount', 'remainder__Profit', 'remainder__Sales_per_Item', 'remainder__Discount_Impact']
Categorical Columns: []


In [235]:
# df["subcat_group"] = df["subcat_group"].replace({
#     "cat__Sub-Category_Phones": "Technology",
#     "cat__Sub-Category_Copiers": "Technology",
#     "cat__Sub-Category_Machines": "Technology",

#     "cat__Sub-Category_Chairs": "Furniture",
#     "cat__Sub-Category_Tables": "Furniture",
#     "cat__Sub-Category_Bookcases": "Furniture",
#     "cat__Sub-Category_Furnishings": "Furniture",

#     "cat__Sub-Category_Binders": "Office Supplies",
#     "cat__Sub-Category_Paper": "Office Supplies",
#     "cat__Sub-Category_Labels": "Office Supplies",
#     "cat__Sub-Category_Envelopes": "Office Supplies",
#     "cat__Sub-Category_Supplies": "Office Supplies",
#     "cat__Sub-Category_Fasteners": "Office Supplies",
#     "cat__Sub-Category_Art": "Office Supplies",
#     "cat__Sub-Category_Appliances": "Office Supplies",
#     "cat__Sub-Category_Storage": "Office Supplies"
# })

In [236]:
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

X_scaled = preprocessor.fit_transform(df)

In [237]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X_scaled)

In [238]:
models = {
"KMeans": KMeans(),
"MiniBatch KMeans": MiniBatchKMeans(),
"Agglomerative": AgglomerativeClustering(),
"DBSCAN": DBSCAN(),
"OPTICS": OPTICS(),
"Gaussian Mixture": GaussianMixture(),
"Birch": Birch()
}


for name, model in models.items():
    
    # Fit model
    if name == "Gaussian Mixture":
        labels = model.fit_predict(X_pca)
    else:
        labels = model.fit_predict(X_pca)

    print(name)
    
    if len(set(labels)) > 1:
        
        silhouette = silhouette_score(X_pca, labels)
        db_index = davies_bouldin_score(X_pca, labels)
        ch_score = calinski_harabasz_score(X_pca, labels)

        print("Model performance")
        print("- Silhouette Score: {:.4f}".format(silhouette))
        print("- Davies-Bouldin Index: {:.4f}".format(db_index))
        print("- Calinski-Harabasz Score: {:.4f}".format(ch_score))
    
    else:
        print("Model produced only one cluster. Metrics cannot be computed.")
    
    print("="*35)
    print("\n")

KMeans
Model performance
- Silhouette Score: 0.7790
- Davies-Bouldin Index: 0.6314
- Calinski-Harabasz Score: 7394.6796


MiniBatch KMeans
Model performance
- Silhouette Score: 0.3569
- Davies-Bouldin Index: 1.0372
- Calinski-Harabasz Score: 1004.0389


Agglomerative
Model performance
- Silhouette Score: 0.9915
- Davies-Bouldin Index: 0.0059
- Calinski-Harabasz Score: 4046.0496


DBSCAN
Model performance
- Silhouette Score: 0.2633
- Davies-Bouldin Index: 1.8316
- Calinski-Harabasz Score: 69.9714




c:\Users\SOHAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


OPTICS
Model performance
- Silhouette Score: -0.0343
- Davies-Bouldin Index: 2.0706
- Calinski-Harabasz Score: 2.3648


Gaussian Mixture
Model produced only one cluster. Metrics cannot be computed.


Birch
Model performance
- Silhouette Score: 0.9488
- Davies-Bouldin Index: 0.4427
- Calinski-Harabasz Score: 4458.1757




In [239]:
unique, counts = np.unique(kmeans_labels, return_counts=True)
print(dict(zip(unique, counts)))

{np.int32(0): np.int64(9944), np.int32(1): np.int64(1), np.int32(2): np.int64(49)}


In [240]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in range(3, 18):
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X_pca)

    score = silhouette_score(X_pca, labels)
    print(k, score)

3 0.9533800343944685
4 0.8738183120668624
5 0.8485144771297448
6 0.8493313774893161
7 0.8371444052844548
8 0.7989660779373455
9 0.7793018068662739
10 0.7736809215692637
11 0.7728010939581065
12 0.7728035037394871
13 0.7161234173343966
14 0.6952272984422317
15 0.675081927942998
16 0.6765704478579616
17 0.6765980951708109


In [241]:
kmeans_params = {
    "n_clusters": [3,4,5,6,7,8,9,10],
    "init": ["k-means++","random"],
    "n_init": [10,20,30],
    "max_iter": [200,300,500]
}

minibatch_params = {
    "n_clusters": [3,4,5,6,7,8,9,10],
    "batch_size": [100,200,500,1000],
    "init": ["k-means++","random"],
    "n_init": [10,20,30],
    "max_iter": [200,300,500]
}

In [242]:
randomcv_models = [
    ("KMeans", KMeans(), kmeans_params),
    ("MiniBatchKMeans", MiniBatchKMeans(), minibatch_params)
]

In [243]:
from sklearn.metrics import silhouette_score

def silhouette_scorer(estimator, X, y=None):
    labels = estimator.fit_predict(X)

    if len(set(labels)) > 1:
        return silhouette_score(X, labels)
    else:
        return -1

In [244]:
from sklearn.model_selection import GridSearchCV

model_param = {}

for name, model, params in randomcv_models:

    grid = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring=silhouette_scorer,
        cv=[(list(range(len(X_pca))), list(range(len(X_pca))))],
        verbose=2,
        n_jobs=2
    )

    grid.fit(X_pca)

    model_param[name] = grid.best_params_

Fitting 1 folds for each of 144 candidates, totalling 144 fits
Fitting 1 folds for each of 576 candidates, totalling 576 fits


In [261]:
for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

---------------- Best Params for KMeans -------------------
{'init': 'random', 'max_iter': 200, 'n_clusters': 3, 'n_init': 10}
---------------- Best Params for MiniBatchKMeans -------------------
{'batch_size': 200, 'init': 'random', 'max_iter': 200, 'n_clusters': 3, 'n_init': 10}


In [262]:
kmeans_final = KMeans(
    n_init=10,
    n_clusters=3,
    max_iter=200,
    init='random',
    random_state=42
)

minibatch_final = MiniBatchKMeans(
    n_init=30,
    n_clusters=3,
    max_iter=500,
    init='random',
    batch_size=200,
    random_state=42
)

kmeans_labels = kmeans_final.fit_predict(X_scaled)
minibatch_labels = minibatch_final.fit_predict(X_scaled)

In [263]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def evaluate_clustering(X, labels):

    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    ch = calinski_harabasz_score(X, labels)

    print("Silhouette Score:", sil)
    print("Davies-Bouldin Index:", db)
    print("Calinski-Harabasz Score:", ch)

In [264]:
print("KMeans Final Model")
evaluate_clustering(X_scaled, kmeans_labels)
print("="*40)
print("MiniBatchKMeans Final Model")
evaluate_clustering(X_scaled, minibatch_labels)
print("="*40)

KMeans Final Model
Silhouette Score: 0.9436348473527796
Davies-Bouldin Index: 0.7066070811168069
Calinski-Harabasz Score: 3824.287371948328
MiniBatchKMeans Final Model
Silhouette Score: 0.013263257475092487
Davies-Bouldin Index: 2.141473634907197
Calinski-Harabasz Score: 786.3288228013504
